
# StreamVault Licensing Analysis

## Data Profiling

**Purpose:** Examine the candidate movie ID source to assess its structure, completeness, validity, and uniqueness before transforming the data.

**Source:** `id_list.csv`

**Profiling date:** 2026-11-19


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "id_list.csv").exists() else Path.cwd().parent
ID_LIST_PATH = PROJECT_ROOT / "id_list.csv"

ids_df = pd.read_csv(
    ID_LIST_PATH,
    dtype=str,
    keep_default_na=False,
)

print("Source file:", ID_LIST_PATH.name)
print("Parsed rows:", len(ids_df))
print("Columns:", ids_df.columns.tolist())

display(ids_df.head(10))
display(ids_df.tail(10))

Source file: id_list.csv
Parsed rows: 126
Columns: ['tmdb_id']


,tmdb_id
0,155
1,165
2,242
3,49026
4,8077
5,331
6,680
7,58
8,10138
9,673


,tmdb_id
116,534
117,330
118,1895
119,272
120,49051
121,166426
122,157336
123,620
124,296
125,8078


In [2]:
# Check for blank and invalid IDs
clean_ids = ids_df["tmdb_id"].str.strip()

blank_mask = clean_ids.eq("")
valid_mask = clean_ids.str.fullmatch(r"[1-9]\d*")
invalid_mask = ~blank_mask & ~valid_mask

# Check duplicates
duplicate_mask = clean_ids.duplicated(keep=False)

print("Total parsed rows:", len(ids_df))
print("Blank IDs:", blank_mask.sum())
print("Valid numeric ID rows:", valid_mask.sum())
print("Invalid nonblank IDs:", invalid_mask.sum())
print("Rows belonging to duplicate groups:", duplicate_mask.sum())
print("Unique valid IDs:", clean_ids[valid_mask].nunique())

print("\nInvalid values:")
display(ids_df.loc[invalid_mask])

print("\nDuplicate rows:")
display(ids_df.loc[duplicate_mask].sort_values("tmdb_id"))

Total parsed rows: 126
Blank IDs: 0
Valid numeric ID rows: 123
Invalid nonblank IDs: 3
Rows belonging to duplicate groups: 12
Unique valid IDs: 117

Invalid values:


,tmdb_id
28,-1
92,tt0468569
108,N/A



Duplicate rows:


,tmdb_id
0,155
88,155
53,1891
87,1891
25,278
58,278
77,550
91,550
38,603
41,603


In [3]:
# Read the source file as physical lines
with ID_LIST_PATH.open("r", encoding="utf-8") as file:
    physical_lines = file.readlines()

blank_line_numbers = [
    line_number
    for line_number, line in enumerate(physical_lines, start=1)
    if not line.strip()
]

print("Physical lines, including header:", len(physical_lines))
print("Physical data lines, excluding header:", len(physical_lines) - 1)
print("Parsed data rows:", len(ids_df))
print("Completely blank physical lines:", blank_line_numbers)

Physical lines, including header: 128
Physical data lines, excluding header: 127
Parsed data rows: 126
Completely blank physical lines: [69]


In [4]:
STATUS_PATH = PROJECT_ROOT / "data" / "processed" / "extraction_status.csv"

status_df = pd.read_csv(
    STATUS_PATH,
    dtype=str,
    keep_default_na=False,
)

print("Status file:", STATUS_PATH.name)
print("Status records:", len(status_df))
print("\nStatus columns:")
print(status_df.columns.tolist())

print("\nMovie extraction outcomes:")
display(status_df["movie_status"].value_counts().rename_axis("status").to_frame("records"))

print("\nLast 10 status records:")
display(status_df.tail(10))

Status file: extraction_status.csv
Status records: 127

Status columns:
['source_row', 'movie_id', 'movie_status', 'credits_status', 'error']

Movie extraction outcomes:


,records
status,
success,112
duplicate_id,6
not_found,5
invalid_id,4



Last 10 status records:


,source_row,movie_id,movie_status,credits_status,error
117,119,534,success,success,
118,120,330,success,success,
119,121,1895,success,success,
120,122,272,success,success,
121,123,49051,success,success,
122,124,166426,success,success,
123,125,157336,success,success,
124,126,620,success,success,
125,127,296,success,success,
126,128,8078,success,success,


In [5]:
# Compare source IDs with extraction status IDs
source_ids = ids_df["tmdb_id"].str.strip()
status_ids = status_df["movie_id"].str.strip()

print("Source parsed rows:", len(source_ids))
print("Status records:", len(status_ids))
print("Blank movie IDs in status:", status_ids.eq("").sum())

print("\nStatus records with blank movie IDs:")
display(status_df.loc[status_ids.eq("")])

print("\nStatus records with invalid IDs:")
display(
    status_df.loc[
        ~status_ids.eq("")
        & ~status_ids.str.fullmatch(r"[1-9]\d*")
    ]
)

Source parsed rows: 126
Status records: 127
Blank movie IDs in status: 1

Status records with blank movie IDs:


,source_row,movie_id,movie_status,credits_status,error
67,69,,invalid_id,invalid_id,



Status records with invalid IDs:


,source_row,movie_id,movie_status,credits_status,error
28,30,-1,invalid_id,invalid_id,
93,95,tt0468569,invalid_id,invalid_id,
109,111,N/A,invalid_id,invalid_id,


In [6]:
blank_status = status_df.loc[status_df["movie_id"].str.strip().eq("")]

display(blank_status.T)


,67
source_row,69
movie_id,
movie_status,invalid_id
credits_status,invalid_id
error,


In [7]:
# Compare the original parsed IDs with the extraction status IDs
source_ids = ids_df["tmdb_id"].str.strip()
status_ids = status_df["movie_id"].str.strip()

print("Unique nonblank source IDs:", source_ids[source_ids.ne("")].nunique())
print("Unique nonblank status IDs:", status_ids[status_ids.ne("")].nunique())

print("\nStatus counts:")
display(status_df["movie_status"].value_counts().to_frame("records"))

print("\nSource IDs that are invalid:")
display(ids_df.loc[
    ~source_ids.eq("") & ~source_ids.str.fullmatch(r"[1-9]\d*")
])

print("\nStatus records marked invalid:")
display(status_df.loc[status_df["movie_status"].eq("invalid_id")])

Unique nonblank source IDs: 120
Unique nonblank status IDs: 120

Status counts:


,records
movie_status,
success,112
duplicate_id,6
not_found,5
invalid_id,4



Source IDs that are invalid:


,tmdb_id
28,-1
92,tt0468569
108,N/A



Status records marked invalid:


,source_row,movie_id,movie_status,credits_status,error
28,30,-1,invalid_id,invalid_id,
67,69,,invalid_id,invalid_id,
93,95,tt0468569,invalid_id,invalid_id,
109,111,N/A,invalid_id,invalid_id,


In [8]:
# Count each nonblank ID in both datasets
source_counts = (
    source_ids[source_ids.ne("")]
    .value_counts()
    .rename("source_count")
)

status_counts = (
    status_ids[status_ids.ne("")]
    .value_counts()
    .rename("status_count")
)

# Compare counts for every ID found in either dataset
id_comparison = pd.concat(
    [source_counts, status_counts],
    axis=1,
).fillna(0).astype(int)

id_comparison["counts_match"] = (
    id_comparison["source_count"]
    == id_comparison["status_count"]
)

print("IDs with matching record counts:",
      id_comparison["counts_match"].sum())

print("IDs with mismatched record counts:",
      (~id_comparison["counts_match"]).sum())

display(id_comparison.loc[~id_comparison["counts_match"]])

IDs with matching record counts: 120
IDs with mismatched record counts: 0


,source_count,status_count,counts_match


In [9]:
print("Credits extraction outcomes:")
display(
    status_df["credits_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .to_frame("records")
)

print("\nRecords where movie and credits outcomes differ:")
display(
    status_df.loc[
        status_df["movie_status"] != status_df["credits_status"],
        ["source_row", "movie_id", "movie_status", "credits_status", "error"]
    ]
)

Credits extraction outcomes:


,records
status,
success,112
duplicate_id,6
not_found,5
invalid_id,4



Records where movie and credits outcomes differ:


,source_row,movie_id,movie_status,credits_status,error


In [10]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"

movie_files = list(RAW_DIR.glob("*movie*.json"))
credits_files = list(RAW_DIR.glob("*credits*.json"))

print("Raw directory:", "data/raw/")
print("Movie JSON files:", len(movie_files))
print("Credits JSON files:", len(credits_files))

print("\nSample movie filenames:")
print([p.name for p in movie_files[:5]])

print("\nSample credits filenames:")
print([p.name for p in credits_files[:5]])

Raw directory: data/raw/
Movie JSON files: 112
Credits JSON files: 112

Sample movie filenames:
['movie_10138.json', 'movie_10192.json', 'movie_10193.json', 'movie_105.json', 'movie_10681.json']

Sample credits filenames:
['credits_10138.json', 'credits_10192.json', 'credits_10193.json', 'credits_105.json', 'credits_10681.json']


In [11]:
movie_file_ids = {
    p.stem.removeprefix("movie_")
    for p in movie_files
}

credits_file_ids = {
    p.stem.removeprefix("credits_")
    for p in credits_files
}

print("Unique movie file IDs:", len(movie_file_ids))
print("Unique credits file IDs:", len(credits_file_ids))

print("\nMovie files without matching credits files:")
print(sorted(movie_file_ids - credits_file_ids))

print("\nCredits files without matching movie files:")
print(sorted(credits_file_ids - movie_file_ids))

print("\nIDs with both files:", len(movie_file_ids & credits_file_ids))

Unique movie file IDs: 112
Unique credits file IDs: 112

Movie files without matching credits files:
[]

Credits files without matching movie files:
[]

IDs with both files: 112


In [12]:
import json

sample_id = sorted(movie_file_ids)[0]

movie_path = RAW_DIR / f"movie_{sample_id}.json"
credits_path = RAW_DIR / f"credits_{sample_id}.json"

with movie_path.open("r", encoding="utf-8") as file:
    movie_json = json.load(file)

with credits_path.open("r", encoding="utf-8") as file:
    credits_json = json.load(file)

print("Sample movie ID:", sample_id)

print("\nMovie JSON top-level keys:")
print(list(movie_json.keys()))

print("\nCredits JSON top-level keys:")
print(list(credits_json.keys()))

print("\nMovie title:", movie_json.get("title"))
print("Release date:", movie_json.get("release_date"))
print("Runtime:", movie_json.get("runtime"))
print("Genres:", movie_json.get("genres"))
print("Collection:", movie_json.get("belongs_to_collection"))

print("\nCast entries:", len(credits_json.get("cast", [])))
print("Crew entries:", len(credits_json.get("crew", [])))

Sample movie ID: 10138

Movie JSON top-level keys:
['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

Credits JSON top-level keys:
['id', 'cast', 'crew']

Movie title: Iron Man 2
Release date: 2010-04-28
Runtime: 124
Genres: [{'id': 12, 'name': 'Adventure'}, {'id': 28, 'name': 'Action'}, {'id': 878, 'name': 'Science Fiction'}]
Collection: {'id': 131292, 'name': 'Iron Man Collection', 'poster_path': '/fbeJ7f0aD4A112Bc1tnpzyn82xO.jpg', 'backdrop_path': '/rI8zOWkRQJdlAyQ6WJOSlYK6JxZ.jpg'}

Cast entries: 86
Crew entries: 159


In [13]:
print("First genre:")
print(movie_json["genres"][0])

print("\nCollection fields:")
print(movie_json["belongs_to_collection"].keys())

print("\nFirst cast member:")
print(credits_json["cast"][0])

print("\nFirst crew member:")
print(credits_json["crew"][0])

print("\nCrew jobs:")
print(
    sorted({
        person.get("job", "")
        for person in credits_json["crew"]
    })
)

First genre:
{'id': 12, 'name': 'Adventure'}

Collection fields:
dict_keys(['id', 'name', 'poster_path', 'backdrop_path'])

First cast member:
{'adult': False, 'gender': 2, 'id': 3223, 'known_for_department': 'Acting', 'name': 'Robert Downey Jr.', 'original_name': 'Robert Downey Jr.', 'popularity': 10.3742, 'profile_path': '/5qHNjhtjMD4YWH3UP0rm4tKwxCL.jpg', 'cast_id': 30, 'character': 'Tony Stark', 'credit_id': '52fe43329251416c75007341', 'order': 0}

First crew member:
{'adult': False, 'gender': 2, 'id': 24192, 'known_for_department': 'Sound', 'name': 'Dave Jordan', 'original_name': 'Dave Jordan', 'popularity': 0.4748, 'profile_path': '/btzUgkKCllxClxIgLhmhZ1qaPw.jpg', 'credit_id': '604a94f990fca3005a8b3709', 'department': 'Sound', 'job': 'Music Supervisor'}

Crew jobs:
['ADR & Dubbing', 'Animation', 'Animation Supervisor', 'Art Direction', 'Associate Producer', 'Camera Operator', 'Casting', 'Co-Producer', 'Comic Book', 'Compositing Supervisor', 'Costume Design', 'Digital Compositor'

In [14]:
director_records = [
    {
        "person_id": person.get("id"),
        "director_name": person.get("name"),
        "job": person.get("job"),
    }
    for person in credits_json["crew"]
    if person.get("job") == "Director"
]

print("Directors found:", len(director_records))
display(pd.DataFrame(director_records))

Directors found: 1


,person_id,director_name,job
0,15277,Jon Favreau,Director


In [15]:
# IDs recorded as successful movie extractions
successful_ids = set(
    status_df.loc[
        status_df["movie_status"].eq("success"),
        "movie_id"
    ].str.strip()
)

# Compare them with the movie JSON files on disk
print("Successful movie status records:", len(successful_ids))
print("Unique successful movie IDs:", len(successful_ids))
print("Movie JSON IDs on disk:", len(movie_file_ids))

print("\nFiles without a successful movie status:")
print(sorted(movie_file_ids - successful_ids))

print("\nSuccessful IDs without a movie JSON file:")
print(sorted(successful_ids - movie_file_ids))

Successful movie status records: 112
Unique successful movie IDs: 112
Movie JSON IDs on disk: 112

Files without a successful movie status:
[]

Successful IDs without a movie JSON file:
[]


In [16]:
print("Status records for movie ID 155:")
display(
    status_df.loc[
        status_df["movie_id"].str.strip().eq("155")
    ]
)

print("\nMovie 155 JSON exists:", (RAW_DIR / "movie_155.json").exists())
print("Credits 155 JSON exists:", (RAW_DIR / "credits_155.json").exists())

movie_155_path = RAW_DIR / "movie_155.json"

if movie_155_path.exists():
    with movie_155_path.open("r", encoding="utf-8") as file:
        movie_155 = json.load(file)

    print("\nMovie 155 payload details:")
    print("Payload ID:", movie_155.get("id"))
    print("Title:", movie_155.get("title"))
    print("Release date:", movie_155.get("release_date"))

Status records for movie ID 155:


,source_row,movie_id,movie_status,credits_status,error
0,2,155,success,success,
89,91,155,duplicate_id,duplicate_id,



Movie 155 JSON exists: True
Credits 155 JSON exists: True

Movie 155 payload details:
Payload ID: 155
Title: The Dark Knight
Release date: 2008-07-16


In [17]:
movie_id_mismatches = []

for path in sorted(RAW_DIR.glob("movie_*.json")):
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    filename_id = path.stem.removeprefix("movie_")
    payload_id = str(payload.get("id", ""))

    if filename_id != payload_id:
        movie_id_mismatches.append({
            "filename": path.name,
            "filename_id": filename_id,
            "payload_id": payload_id,
        })

print("Movie JSON files checked:", len(movie_files))
print("Filename/payload ID mismatches:", len(movie_id_mismatches))
display(pd.DataFrame(movie_id_mismatches))

Movie JSON files checked: 112
Filename/payload ID mismatches: 0


""


In [18]:
credits_id_mismatches = []

for path in sorted(RAW_DIR.glob("credits_*.json")):
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    filename_id = path.stem.removeprefix("credits_")
    payload_id = str(payload.get("id", ""))

    if filename_id != payload_id:
        credits_id_mismatches.append({
            "filename": path.name,
            "filename_id": filename_id,
            "payload_id": payload_id,
        })

print("Credits JSON files checked:", len(credits_files))
print("Filename/payload ID mismatches:", len(credits_id_mismatches))
display(pd.DataFrame(credits_id_mismatches))

Credits JSON files checked: 112
Filename/payload ID mismatches: 0


""


In [19]:
required_movie_keys = {
    "id",
    "title",
    "release_date",
    "runtime",
    "revenue",
    "genres",
    "belongs_to_collection",
}

required_credits_keys = {"id", "cast", "crew"}

movie_structure_issues = []
credits_structure_issues = []

for path in sorted(RAW_DIR.glob("movie_*.json")):
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    missing_keys = required_movie_keys - payload.keys()

    if missing_keys:
        movie_structure_issues.append({
            "filename": path.name,
            "missing_keys": sorted(missing_keys),
        })

    if not isinstance(payload.get("genres"), list):
        movie_structure_issues.append({
            "filename": path.name,
            "issue": "genres is not a list",
        })

    collection = payload.get("belongs_to_collection")
    if collection is not None and not isinstance(collection, dict):
        movie_structure_issues.append({
            "filename": path.name,
            "issue": "belongs_to_collection is neither a dictionary nor None",
        })

for path in sorted(RAW_DIR.glob("credits_*.json")):
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    missing_keys = required_credits_keys - payload.keys()

    if missing_keys:
        credits_structure_issues.append({
            "filename": path.name,
            "missing_keys": sorted(missing_keys),
        })

    for key in ("cast", "crew"):
        if not isinstance(payload.get(key), list):
            credits_structure_issues.append({
                "filename": path.name,
                "issue": f"{key} is not a list",
            })

print("Movie structure issues:", len(movie_structure_issues))
display(pd.DataFrame(movie_structure_issues))

print("\nCredits structure issues:", len(credits_structure_issues))
display(pd.DataFrame(credits_structure_issues))

Movie structure issues: 0


""



Credits structure issues: 0


""


In [20]:
movie_field_rows = []

for path in sorted(RAW_DIR.glob("movie_*.json")):
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    movie_field_rows.append({
        "movie_id": payload.get("id"),
        "title": payload.get("title"),
        "release_date": payload.get("release_date"),
        "runtime": payload.get("runtime"),
        "revenue": payload.get("revenue"),
        "vote_average": payload.get("vote_average"),
        "vote_count": payload.get("vote_count"),
        "collection_id": (
            payload.get("belongs_to_collection") or {}
        ).get("id"),
        "collection_name": (
            payload.get("belongs_to_collection") or {}
        ).get("name"),
    })

movie_fields_df = pd.DataFrame(movie_field_rows)

print("Movies profiled:", len(movie_fields_df))
print("\nMissing or empty values by field:")

missing_counts = (
    movie_fields_df.isna()
    | movie_fields_df.astype("string").apply(
        lambda col: col.str.strip().eq("")
    )
).sum()

display(missing_counts.rename("missing_or_empty").to_frame())

print("\nSample records with missing release dates:")
display(
    movie_fields_df.loc[
        movie_fields_df["release_date"].isna()
        | movie_fields_df["release_date"].astype("string").str.strip().eq(""),
        ["movie_id", "title", "release_date"]
    ].head(10)
)

Movies profiled: 112

Missing or empty values by field:


,missing_or_empty
movie_id,0
title,0
release_date,0
runtime,0
revenue,0
vote_average,0
vote_count,0
collection_id,16
collection_name,16



Sample records with missing release dates:


,movie_id,title,release_date


In [21]:
has_collection = movie_fields_df["collection_id"].notna()

print("Movies profiled:", len(movie_fields_df))
print("Movies with a collection:", has_collection.sum())
print("Movies without a collection:", (~has_collection).sum())

print("\nMovies without a collection:")
display(
    movie_fields_df.loc[
        ~has_collection,
        ["movie_id", "title", "release_date", "revenue"]
    ].sort_values("title")
)

print("\nCollection names with the most candidate movies:")
display(
    movie_fields_df.loc[has_collection]
    .groupby("collection_name")
    .agg(
        movie_count=("movie_id", "nunique"),
        titles=("title", lambda values: ", ".join(values))
    )
    .sort_values("movie_count", ascending=False)
    .head(15)
)

Movies profiled: 112
Movies with a collection: 96
Movies without a collection: 16

Movies without a collection:


,movie_id,title,release_date,revenue
69,550,Fight Club,1999-10-15,100853753
15,13,Forrest Gump,1994-06-23,677387716
91,769,GoodFellas,1990-09-12,47072327
47,27205,Inception,2010-07-15,839030630
21,157336,Interstellar,2014-11-04,746606706
92,77,Memento,2000-10-11,40060108
87,680,Pulp Fiction,1994-09-10,213928762
80,641,Requiem for a Dream,2000-10-06,7390108
6,111,Scarface,1983-12-09,66023329
63,424,Schindler's List,1993-12-15,321365567



Collection names with the most candidate movies:


,movie_count,titles
collection_name,,
Harry Potter Collection,8,"Harry Potter and the Deathly Hallows: Part 1, ..."
Star Wars Collection,6,"Star Wars, The Empire Strikes Back, Return of ..."
Pirates of the Caribbean Collection,5,Pirates of the Caribbean: Dead Men Tell No Tal...
Mission: Impossible Collection,5,"Mission: Impossible - Rogue Nation, Mission: I..."
Die Hard Collection,4,"Live Free or Die Hard, Die Hard: With a Vengea..."
Shrek Collection,4,"Shrek Forever After, Shrek, Shrek 2, Shrek the..."
Indiana Jones Collection,4,Indiana Jones and the Kingdom of the Crystal S...
Alien Collection,4,"Alien, Aliens, Alien³, Alien Resurrection"
The Avengers Collection,4,"The Avengers, Avengers: Endgame, Avengers: Inf..."
